In [4]:
import sys

sys.path.append("/Users/liu.9639/Downloads/A2")

import torch
import matplotlib.pyplot as plt
from a2.models.transformer_seq2seq import *
from a2.data import *
# Import established script functions and configs from your repository
from train_transformer import *

from train_mlp import *



"--epochs", "10",
"--lr","3e-4"
"--num-layers", "2",
"--hidden-dim", "128",
Test  | loss 1.3343 acc 0.0059

"--epochs", "20",
"--lr","3e-4"
"--num-layers", "2",
"--hidden-dim", "128",
Test  | loss 1.1636 acc 0.0187

"--epochs", "20",
"--lr","3e-4"
"--num-layers", "2",
"--hidden-dim", "128",
Test  | loss 0.3976 acc 0.4106

"--epochs", "30",
"--lr","3e-4"
"--num-layers", "2",
"--hidden-dim", "128",
Test  | loss 0.2359 acc 0.5903

"--epochs", "40",
0.1748 acc 0.7172

"--epochs", "50",
Test  | loss 0.1140 acc 0.8655


"--epochs", "80",
loss 0.0482 acc 0.9443

"--epochs", "10",
"--lr","1e-3",
"--num-layers", "2",
"--hidden-dim", "128",
Test  | loss 0.4042 acc 0.4411

In [5]:
import argparse

# Define the parser
parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=10)
parser.add_argument("--batch-size", type=int, default=128)
parser.add_argument("--hidden-dim", type=int, default=128)
parser.add_argument("--num-layers", type=int, default=2)
parser.add_argument("--input-mode", choices=["onehot", "scalar"], default="onehot", help="onehot: one-hot digits of a and b (raw input); scalar: hand-crafted numeric features")
parser.add_argument("--no-digit-feat", action="store_true", help="scalar mode: disable digit count features")
parser.add_argument("--normalize", action="store_true", help="scalar mode: normalize input features")
parser.add_argument("--lr", type=float, default=3e-4)
parser.add_argument("--max-digits", type=int, default=3)
parser.add_argument("--test-max-digits", type=int, default=None, help="Max digits for the test set (default: --max-digits); set larger to test generalization to longer numbers")
parser.add_argument("--train-size", type=int, default=20000)
parser.add_argument("--val-size", type=int, default=2000)
parser.add_argument("--test-size", type=int, default=2000)
parser.add_argument("--weight-decay", type=float, default=0.01, help="AdamW weight decay")
parser.add_argument("--seed", type=int, default=42, help="Seed for model init / batch order")
parser.add_argument("--data-seed", type=int, default=0, help="Seed for the train/val/test data (and the label permutation)")
parser.add_argument("--random-labels", action="store_true", help="Shuffle targets across the training set (Zhang et al. 2017)")

# Define your argument list for Jupyter Notebook
args_list = [
    "--epochs", "10",
    "--lr","1e-3",
    "--num-layers", "2",
    "--hidden-dim", "128",
    "--max-digits", "3",
    "--train-size", "50000",
    "--val-size", "2000",
    "--test-size", "2000",
]

# Parse the argument list
args = parser.parse_args(args_list)

# Access your arguments as usual
print(args)

Namespace(epochs=10, batch_size=128, hidden_dim=128, num_layers=2, input_mode='onehot', no_digit_feat=False, normalize=False, lr=0.001, max_digits=3, test_max_digits=None, train_size=50000, val_size=2000, test_size=2000, weight_decay=0.01, seed=42, data_seed=0, random_labels=False)


In [6]:
set_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = default_vocab()

train_ex = generate_dataset(args.train_size, args.max_digits, seed=args.data_seed)
val_ex = generate_dataset(args.val_size, args.max_digits, seed=args.data_seed + 1)
test_max_digits = args.test_max_digits or args.max_digits
test_ex = generate_dataset(args.test_size, test_max_digits, seed=args.data_seed + 2)
if args.random_labels:
    train_ex = permute_targets(train_ex, seed=args.data_seed)
    print("Training targets randomly permuted (--random-labels); val/test are unchanged.")

train_ds = AdditionDataset(train_ex, vocab)
val_ds = AdditionDataset(val_ex, vocab)
test_ds = AdditionDataset(test_ex, vocab)

collate = lambda b: collate_fn(b, pad_id=vocab.pad_id)  # noqa: E731
train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate)
train_eval_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)

print(f"Datasets: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}")

model = MLPAddition(
    MLPConfig(
        vocab_size=vocab.size,
        # Size the input for the longest operands it will see; with --test-max-digits >
        # --max-digits, the extra (most significant) digit slots are always 0 in training.
        max_digits=max(args.max_digits, test_max_digits),
        hidden_dim=args.hidden_dim,
        num_layers=args.num_layers,
        input_mode=args.input_mode,
        digit_feat=not args.no_digit_feat,
        normalize=args.normalize,
    ),
    pad_id=vocab.pad_id,
    sos_id=vocab.sos_id,
    eos_id=vocab.eos_id,
    plus_id=vocab.stoi["+"],
    digit0_id=vocab.stoi["0"],
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

best_val = 0.0
for epoch in range(1, args.epochs + 1):
    print(f"\nEpoch {epoch}/{args.epochs}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    # train_acc above is a running average over minibatches *during* the epoch; re-evaluate
    # the final weights on the whole training set for a clean train exact-match number.
    _, train_em = evaluate(model, train_eval_loader, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} (full-set EM {train_em:.4f}) | val loss {val_loss:.4f} acc {val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test  | loss {test_loss:.4f} acc {test_acc:.4f}")

Datasets: train=50000 val=2000 test=2000
Batches: train=391 val=16 test=16

Epoch 1/10


Epoch 01 | train loss 1.5827 acc 0.0025 (full-set EM 0.0046) | val loss 1.3769 acc 0.0063

Epoch 2/10


Epoch 02 | train loss 1.2948 acc 0.0069 (full-set EM 0.0093) | val loss 1.2192 acc 0.0123

Epoch 3/10


Epoch 03 | train loss 1.1426 acc 0.0202 (full-set EM 0.0278) | val loss 1.0634 acc 0.0270

Epoch 4/10


Epoch 04 | train loss 1.0018 acc 0.0410 (full-set EM 0.0488) | val loss 0.9432 acc 0.0490

Epoch 5/10


Epoch 05 | train loss 0.8891 acc 0.0598 (full-set EM 0.0722) | val loss 0.8437 acc 0.0642

Epoch 6/10


... (2 outputs left out)

Epoch 08 | train loss 0.5775 acc 0.2321 (full-set EM 0.2773) | val loss 0.5433 acc 0.2688

Epoch 9/10


Epoch 09 | train loss 0.5076 acc 0.2982 (full-set EM 0.3570) | val loss 0.4765 acc 0.3492

Epoch 10/10


Epoch 10 | train loss 0.4347 acc 0.4037 (full-set EM 0.4629) | val loss 0.3952 acc 0.4572


Test  | loss 0.4042 acc 0.4411
